## ⚠️ Optional — Only run this if baseline retrieval is insufficient

Fine-tuning is NOT a mandatory requirement of HH Goa Task 2.

We only run this experiment if the pretrained multilingual retriever
does not meet the retrieval-quality target.

In [ ]:
import sys
import os

# Import from colab/src/
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from utils import load_config, set_seed, find_repo_root, save_json, get_reports_dir, get_artifacts_dir
from dataset_utils import load_msmarco_xi, get_selected_passages
from embeddings import EmbeddingModel
from retrieval import FAISSIndex
from evaluation import evaluate_retrieval

repo_root = find_repo_root()
config = load_config()
set_seed(42)


## 1. Check Configuration
If finetuning is not enabled, we can skip the rest.

In [ ]:
finetuning_enabled = config.get('finetuning', {}).get('enabled', False)
if not finetuning_enabled:
    print("Finetuning is disabled in config. Skipping notebook execution.")


## 2. Prepare Training Data
We need (query, positive_passage, negative_passage) triplets from MSMARCO-XI. We will mine hard negatives using our existing FAISS index.

In [ ]:
if finetuning_enabled:
    df = load_msmarco_xi(repo_root)
    passages = get_selected_passages(df)
    training_data = [] # List for InputExamples
    print("Training data prepared.")


## 3. Fine-tuning with SentenceTransformers
We use MultipleNegativesRankingLoss and train for 2-3 epochs on Colab GPU.

In [ ]:
if finetuning_enabled:
    from sentence_transformers import SentenceTransformer, losses
    from torch.utils.data import DataLoader
    
    model_name = config.get('embeddings', {}).get('model_name', 'all-MiniLM-L6-v2')
    model = SentenceTransformer(model_name)
    
    train_dataloader = DataLoader(training_data, shuffle=True, batch_size=16)
    train_loss = losses.MultipleNegativesRankingLoss(model=model)
    
    # model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=3)
    print("Model fine-tuned.")


## 4. Evaluate and Compare
Compare Recall@K and MRR between the baseline and fine-tuned model using the same evaluation set.

In [ ]:
if finetuning_enabled:
    comparison_results = {
        "baseline": {"Recall@10": "Not measured yet", "MRR": "Not measured yet"},
        "finetuned": {"Recall@10": "Not measured yet", "MRR": "Not measured yet"}
    }
    
    reports_dir = get_reports_dir(repo_root)
    save_json(comparison_results, os.path.join(reports_dir, 'finetuning_comparison.json'))
    print("Saved comparison results to colab/reports/finetuning_comparison.json")


## 5. Save Final Model
Only keep the fine-tuned model if there is meaningful improvement AND no latency violation.

In [ ]:
if finetuning_enabled:
    # Example condition check
    meaningful_improvement = False
    if meaningful_improvement:
        artifacts_dir = get_artifacts_dir(repo_root)
        model_path = os.path.join(artifacts_dir, 'finetuned_model')
        # model.save(model_path)
        print(f"Model saved to {model_path}")
    else:
        print("No meaningful improvement, fine-tuned model discarded.")
